# Diarization: several systems and their consensus
Who speaks when, for the wavs of `data/raw_audios/` that `pipeline_paths.py` picks (`AUDIOS`). Every system writes into its own folder.

- **Reference:** `02_chunk_transcribe` cuts the turns by one diarization, the one `DIARIZATION` in `pipeline_paths.py` names: **`nemotron`**, the only system here built for up to 8 speakers.
- **Verifiers:** the other systems check it. `06_finalize` asks each of them what it hears inside every final chunk: the same speaker as the label says, or another speaker inside.
- **Consensus:** DOVER-Lap's vote of all the systems is made too, for comparison. 02 does not cut by it. On the test series (8+ actors, music) it had only 4 speakers, because two systems can't hold more than 4 and their votes merged actors.

**Kernel: `.venv-nemo`** (NeMo, Python 3.12). On the Mac, the project's `.venv` has NeMo too.

**Systems** (`METHODS`). All of them are NVIDIA models that download without a login; pyannote's models need a Hugging Face login and are not used.

| folder | model | how | speakers |
|---|---|---|---|
| `nemotron` | `nvidia/Nemotron-3-Diarization` | end-to-end (Sortformer), streaming | up to 8 |
| `sortformer_v2` | `nvidia/diar_streaming_sortformer_4spk-v2` | end-to-end (Sortformer), streaming | up to 4 |
| `sortformer_v2_1` | `nvidia/diar_streaming_sortformer_4spk-v2.1` | end-to-end (Sortformer), streaming | up to 4 |
| `clustering` | NeMo `ClusteringDiarizer` **guided by `GUIDE`**: speech regions and speaker count from Nemotron, then TitaNet embeddings at three scales + spectral clustering decide who is who | clustering | as many as Nemotron |
| `consensus` | DOVER-Lap of the systems above, for comparison: their labels are matched, then every moment goes to the speakers most systems hear | voting | |

- **4-speaker models:** they merge speakers when there are more than four. As verifiers that does no harm, because 06 counts a system as agreeing when it puts the chunk where it puts the rest of that speaker's speech.
- **Why the clustering is guided:** run on its own, it called the whole test series one speaker. Its own VAD took the music for speech (39 min instead of about 23), and even on Nemotron's speech its speaker counting failed. Guided, it only answers who is who, by voice, which is the question the check needs.

**Per audio**, in `outputs/diarization/<name>/<system>/`:
- `<name>.rttm`: the turns (`start`, `duration`, `speaker`);
- `stats.json`: speech, overlap and silence, plus per speaker: seconds, share of the speech, turns and median turn length;
- `timeline.png`: who speaks when.

Also `outputs/diarization/<name>/agreement.json`: how much each two systems agree. For each pair, every label of one is matched to the label of the other it overlaps most (several may match the same one, so a 4-speaker model is not punished for merging). The value is the share of the speech, where both hear one speaker, that falls on matched labels, averaged over both directions. 1.0 means they split the speech the same way. Test run: 1.0 on the podcast (one speaker), 0.4–0.7 on the series, so the series' speakers are uncertain.

Each system needs 16 kHz mono, so a temporary 16 kHz copy is made once per audio and deleted afterwards. Systems that already have an RTTM are skipped unless `FORCE = True`; the consensus is rebuilt every time.

In [1]:
import gc
import importlib
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import time
import traceback
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch

import pipeline_paths as P
importlib.reload(P)   # a change of RUN in pipeline_paths.py counts without restarting the kernel

root = P.ROOT
out_root = P.OUTPUTS
print(P.describe(), f"| 02 will use: {P.DIARIZATION}")
wavs = P.audio_files()
P.check_sample_rate(wavs)   # 48 kHz from input to output: stops on a wav that is not
for i, p in enumerate(wavs):
    print(i, p.name)

run 'test': reading and writing in /home/intern/projects/tts/runs/test | 02 will use: nemotron
all 2 input wavs are 48000 Hz
0 neyrobiologiya_va_qaysi_kunlik_WqD84GbTcBM.wav
1 shaytanat_qism_milliy_serial_AsthRnnwvD4.wav


In [2]:
ONLY = None             # None = every audio; or a list of indices from the list above, e.g. [3, 5]
FORCE = False           # True = run the systems again even where they have an RTTM
METHODS = ["nemotron", "sortformer_v2", "sortformer_v2_1", "clustering"]
MAX_SPEAKERS = 8        # clustering without a guide: the most speakers it looks for
GUIDE = "nemotron"      # clustering takes its speech regions and speaker count from this system (None: its own VAD and count)
DOVER_STD = 0.1         # DOVER-Lap smoothing (s) before voting; small for quick turn-taking, as its docs advise

SORTFORMER = {          # Sortformer models and their streaming settings (in 80 ms frames; ~30 s latency, from the model cards)
    "nemotron": ("nvidia/Nemotron-3-Diarization",
                 dict(chunk_len=340, chunk_right_context=40, fifo_len=40, spkcache_update_period=300, spkcache_len=264)),
    "sortformer_v2": ("nvidia/diar_streaming_sortformer_4spk-v2",
                      dict(chunk_len=340, chunk_right_context=40, fifo_len=40, spkcache_update_period=300, spkcache_len=188)),
    "sortformer_v2_1": ("nvidia/diar_streaming_sortformer_4spk-v2.1",
                        dict(chunk_len=340, chunk_right_context=40, fifo_len=40, spkcache_update_period=300, spkcache_len=188)),
}
CLUSTERING_CONFIG = root / "pipeline" / "configs" / "diar_infer_general.yaml"   # NeMo's general inference config
assert set(METHODS) <= set(SORTFORMER) | {"clustering"}, METHODS
assert not GUIDE or GUIDE in METHODS[:METHODS.index("clustering")] if "clustering" in METHODS else True, "GUIDE must run before clustering"

In [3]:
RES = 100  # 10 ms grid for stats and agreement


def write_rttm(turns, name, path):
    file_id = "_".join(name.split())  # RTTM fields are separated by spaces: no spaces in the id
    tmp = path.with_suffix(".rttm.tmp")
    with open(tmp, "w") as f:
        for s, e, spk in turns:
            f.write(f"SPEAKER {file_id} 1 {s:.3f} {e - s:.3f} <NA> <NA> {spk} <NA> <NA>\n")
    os.replace(tmp, path)  # an interrupted run can't leave half an RTTM


def read_rttm(path):
    turns = []
    for line in path.read_text().splitlines():
        p = line.split()  # counted from the end, so an id with spaces still works
        s = float(p[-7])
        turns.append((s, s + float(p[-6]), p[-3]))
    return sorted(turns)


def activity(turns, n):
    # speakers, and a (speakers x frames) matrix: who is talking in every 10 ms frame
    speakers = sorted({spk for *_, spk in turns})
    active = np.zeros((len(speakers), n), dtype=bool)
    for s, e, spk in turns:
        active[speakers.index(spk), max(0, round(s * RES)):min(n, round(e * RES))] = True
    return speakers, active


def diarization_stats(audio_name, total_sec, turns):
    n = math.ceil(total_sec * RES)
    speakers, active = activity(turns, n)
    count = active.sum(axis=0)
    speech, overlap = int((count >= 1).sum()), int((count >= 2).sum())
    per_speaker = {}
    for k, spk in enumerate(speakers):
        turn_len = [e - s for s, e, x in turns if x == spk]
        per_speaker[spk] = {
            "seconds": round(active[k].sum() / RES, 2),
            "share_of_speech_pct": round(100 * active[k].sum() / max(speech, 1), 1),  # overlap counts for both
            "turns": len(turn_len),
            "median_turn_sec": round(float(np.median(turn_len)), 2),
        }
    return {
        "audio": audio_name, "total_seconds": round(total_sec, 2), "speakers": len(speakers), "turns": len(turns),
        "speech_seconds": round(speech / RES, 2), "speech_pct": round(100 * speech / n, 1),
        "overlap_seconds": round(overlap / RES, 2), "overlap_pct": round(100 * overlap / n, 1),
        "silence_seconds": round((n - speech) / RES, 2), "silence_pct": round(100 * (n - speech) / n, 1),
        "per_speaker": per_speaker,
    }


def plot_timeline(turns, title, path):
    speakers = sorted({spk for *_, spk in turns})
    fig, ax = plt.subplots(figsize=(12, 0.6 * max(1, len(speakers)) + 1))
    for row, spk in enumerate(speakers):
        ax.broken_barh([(s / 60, (e - s) / 60) for s, e, x in turns if x == spk], (row - 0.4, 0.8))
    ax.set_yticks(range(len(speakers)), speakers)
    ax.set_xlabel("min")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=100)
    plt.close(fig)


def agreement(turns_a, turns_b, n):
    # of the speech where both systems hear exactly one speaker: the share on labels that match, when every label is
    # matched to the other system's label it overlaps most (many to one, both ways, averaged)
    sa, aa = activity(turns_a, n)
    sb, ab = activity(turns_b, n)
    both = (aa.sum(0) == 1) & (ab.sum(0) == 1)
    if not both.any():
        return None
    co = np.zeros((len(sa), len(sb)))
    np.add.at(co, (aa.argmax(0)[both], ab.argmax(0)[both]), 1)
    return round(float((co.max(0).sum() + co.max(1).sum()) / 2 / both.sum()), 3)


_models = {}


def sortformer(method):
    # the loaded model of a Sortformer system (one at a time, to keep the GPU free for the others)
    if method not in _models:
        for k in list(_models):
            del _models[k]
        gc.collect()
        torch.cuda.empty_cache()
        from nemo.collections.asr.models import SortformerEncLabelModel
        name, streaming = SORTFORMER[method]
        model = SortformerEncLabelModel.from_pretrained(name).eval()
        for k, v in streaming.items():
            setattr(model.sortformer_modules, k, v)
        model.sortformer_modules._check_streaming_parameters()
        _models[method] = model
    return _models[method]


def run_system(method, wav16, tmp_dir, guide=None):
    # (start, end, speaker) turns of one system on the 16 kHz copy; guide: turns the clustering takes its speech
    # regions and speaker count from
    if method in SORTFORMER:
        segments = sortformer(method).diarize(audio=[str(wav16)], batch_size=1)
        return sorted((float(s), float(e), spk) for s, e, spk in (line.split() for line in segments[0]))
    from nemo.collections.asr.models import ClusteringDiarizer
    from omegaconf import OmegaConf
    cfg = OmegaConf.load(CLUSTERING_CONFIG)
    manifest = tmp_dir / "manifest.json"
    rttm = None
    if guide:
        rttm = tmp_dir / "guide.rttm"
        rttm.write_text("".join(f"SPEAKER {wav16.stem} 1 {s:.3f} {e - s:.3f} <NA> <NA> {k} <NA> <NA>\n" for s, e, k in guide))
        cfg.diarizer.oracle_vad = True                          # speech regions from the guide
        cfg.diarizer.clustering.parameters.oracle_num_speakers = True   # and its number of speakers
    manifest.write_text(json.dumps(dict(audio_filepath=str(wav16), offset=0, duration=None, label="infer", text="-",
                                        num_speakers=len({k for *_, k in guide}) if guide else None,
                                        rttm_filepath=str(rttm) if rttm else None, uem_filepath=None)) + "\n")
    cfg.diarizer.manifest_filepath = str(manifest)
    cfg.diarizer.out_dir = str(tmp_dir / "clustering")
    cfg.diarizer.clustering.parameters.max_num_speakers = MAX_SPEAKERS
    cfg.verbose = False
    cfg.num_workers = 0
    ClusteringDiarizer(cfg=cfg).diarize()
    return read_rttm(tmp_dir / "clustering" / "pred_rttms" / f"{wav16.stem}.rttm")

## Run every system, then the consensus

In [4]:
selected = wavs if ONLY is None else [wavs[i] for i in ONLY]
dover = Path(sys.executable).parent / "dover-lap"
failed = {}
for n_audio, audio_path in enumerate(selected, 1):
    name = audio_path.stem
    base = out_root / "diarization" / name
    total_sec = sf.info(str(audio_path)).duration
    print(f"[{n_audio}/{len(selected)}] {audio_path.name} ({total_sec / 60:.1f} min)")
    tmp_dir = Path(tempfile.mkdtemp(prefix="diar_", dir=base.parent if base.parent.exists() else None))
    try:
        wav16 = tmp_dir / "audio.wav"   # a plain name: NeMo's clustering uses the file name as its id
        todo = [m for m in METHODS if FORCE or not (base / m / f"{name}.rttm").exists()]
        if todo:
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(audio_path),
                            "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le", str(wav16)], check=True)
        for m in METHODS:
            path = base / m / f"{name}.rttm"
            path.parent.mkdir(parents=True, exist_ok=True)
            if m in todo:
                t0 = time.time()
                try:
                    g = base / str(GUIDE) / f"{name}.rttm"
                    guide = read_rttm(g) if m == "clustering" and GUIDE and g.exists() else None
                    write_rttm(run_system(m, wav16, tmp_dir, guide), name, path)
                    print(f"  {m:16} ran in {time.time() - t0:5.0f} s")
                except Exception:
                    failed[f"{name} / {m}"] = traceback.format_exc()
                    print(f"  {m:16} FAILED (the consensus goes on without it)")
                    continue
            turns = read_rttm(path)
            stats = diarization_stats(audio_path.name, total_sec, turns)
            (path.parent / "stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
            plot_timeline(turns, f"{audio_path.name}: {m}", path.parent / "timeline.png")
        # consensus of the systems that have an RTTM
        inputs = [base / m / f"{name}.rttm" for m in METHODS if (base / m / f"{name}.rttm").exists()]
        cons = base / "consensus" / f"{name}.rttm"
        cons.parent.mkdir(parents=True, exist_ok=True)
        if len(inputs) >= 2:
            tmp_out = cons.with_suffix(".tmp.rttm")
            subprocess.run([str(dover), "--gaussian-filter-std", str(DOVER_STD), "--label-mapping", "hungarian",   # its greedy
                            str(tmp_out), *map(str, inputs)],                       # mapping fails on numpy 2
                           check=True, capture_output=True)
            write_rttm(read_rttm(tmp_out), name, cons)   # same file id and format as the others
            tmp_out.unlink()
        else:
            shutil.copy2(inputs[0], cons)
        turns = read_rttm(cons)
        stats = diarization_stats(audio_path.name, total_sec, turns)
        (cons.parent / "stats.json").write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
        plot_timeline(turns, f"{audio_path.name}: consensus", cons.parent / "timeline.png")
        # how much the systems agree with each other and with the consensus
        n = math.ceil(total_sec * RES)
        names = [p.parent.name for p in inputs] + ["consensus"]
        all_turns = {k: read_rttm(base / k / f"{name}.rttm") for k in names}
        table = {a: {b: agreement(all_turns[a], all_turns[b], n) for b in names if b != a} for a in names}
        (base / "agreement.json").write_text(json.dumps(table, indent=2), encoding="utf-8")
        print(f"  {'':16} speakers  speech   " + " ".join(f"{k[:11]:>11}" for k in names))
        for a in names:
            s = json.loads((base / a / "stats.json").read_text())
            print(f"  {a:16} {s['speakers']:8d} {s['speech_seconds'] / 60:5.1f} min "
                  + " ".join(f"{'—' if b == a else table[a][b]:>11}" for b in names))
    except Exception:
        failed[name] = traceback.format_exc()
        print("  FAILED (continuing)")
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
for k in list(_models):
    del _models[k]
gc.collect()
torch.cuda.empty_cache()
print(f"\ndone: {len(selected)} audios; failed: {len(failed)}")
for k, tb in failed.items():
    print("\n" + k + "\n" + tb)

[1/2] neyrobiologiya_va_qaysi_kunlik_WqD84GbTcBM.wav (43.1 min)
[NeMo I 2026-11-27 15:58:01 save_restore_connector:309] Model SortformerEncLabelModel was successfully restored from /home/intern/.cache/huggingface/hub/models--nvidia--Nemotron-3-Diarization/snapshots/52a432672732425e4c19765b9fec9b5810e6f4ba/Nemotron-3-Diarization.nemo.
[NeMo I 2026-11-27 15:58:01 vad_utils:90] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.
  nemotron         ran in    16 s
[NeMo I 2026-11-27 15:58:15 save_restore_connector:309] Model SortformerEncLabelModel was successfully restored from /home/intern/.cache/huggingface/hub/models--nvidia--diar_streaming_sortformer_4spk-v2/snapshots/84edd514b8ef68004c10086918cd62f2148cbd59/diar_streaming_sortformer_4spk-v2.nemo.
[NeMo I 2026-11-27 15:58:15 vad_utils:90] No postprocessing YAML file has been provided. Default postprocessing configurations will be applied.
  sortformer_v2    ran in    18 s
[NeMo I 2026-1

## One audio in detail
The agreement table above: the share of the speech where two systems hear the same speaker. Below, the consensus timeline and its stats.

In [5]:
from IPython.display import Image, display

i = 0  # index from the list at the top
base = out_root / "diarization" / wavs[i].stem
print(json.dumps(json.loads((base / "agreement.json").read_text()), indent=2))
for m in METHODS + ["consensus"]:
    if (base / m / "timeline.png").exists():
        display(Image(str(base / m / "timeline.png")))

{
  "nemotron": {
    "sortformer_v2": 1.0,
    "sortformer_v2_1": 1.0,
    "clustering": 1.0,
    "consensus": 1.0
  },
  "sortformer_v2": {
    "nemotron": 1.0,
    "sortformer_v2_1": 1.0,
    "clustering": 1.0,
    "consensus": 1.0
  },
  "sortformer_v2_1": {
    "nemotron": 1.0,
    "sortformer_v2": 1.0,
    "clustering": 1.0,
    "consensus": 1.0
  },
  "clustering": {
    "nemotron": 1.0,
    "sortformer_v2": 1.0,
    "sortformer_v2_1": 1.0,
    "consensus": 1.0
  },
  "consensus": {
    "nemotron": 1.0,
    "sortformer_v2": 1.0,
    "sortformer_v2_1": 1.0,
    "clustering": 1.0
  }
}
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
